# LLM Evaluation Pipeline – Separat vs. Kombiniert
Vergleicht zwei Prompt-Strategien über mehrere Modelle:
- **Separat**: drei einzelne Prompts pro Aufgabe
- **Kombiniert**: ein Prompt für alle drei Aufgaben

Metriken: Accuracy, Precision, Recall, F1-Score

## 1. Installation

In [ ]:
#%pip install -q langchain langchain-openai langchain-huggingface langchain-community
#%pip install -q transformers torch accelerate scikit-learn pandas pydantic tiktoken

## 2. Datensatz laden

In [50]:
import pandas as pd
from pathlib import Path

repo = Path(".").resolve().parent
df   = pd.read_csv(repo / "data/processed/control/gs_merged.csv")

TASK_MAP = {
    1: "Informationssuche und Verständnis",
    2: "Schreiben und Textarbeit",
    3: "Praktische Unterstützung und Strukturierung",
    4: "Technische und analytische Unterstützung",
    5: "Lernen und Prüfungsvorbereitung",
}
SENTIMENT_MAP = {1: "Unfreundlich", 2: "Neutral", 3: "Freundlich"}
CRITICAL_MAP  = {0: "Nein", 1: "Ja"}

#JSON-Export Rows direkt aus df entfernen
JSON_PATTERN = "Gebe mir den gesamten sichtbaren bisherigen Dialog zwischen uns als JSON"

vorher = len(df)
df = df[~df["content"].str.contains(JSON_PATTERN, na=False)].reset_index(drop=True)
print(f"Entfernte Rows: {vorher - len(df)} | Verbleibend: {len(df)}")

#Chats zusammenführen
df_chats = (
    df.groupby("chat_id")
    .agg(
        text            = ("content", lambda x: "\n".join(
                            f"User_Nachricht_{i+1}: {msg}"
                            for i, msg in enumerate(x))),
        label_task      = ("task",      "first"),
        label_sentiment = ("sentiment", "first"),
        label_critical  = ("critical",  "first"),
    )
    .reset_index()
)

df_chats["label_task"]      = df_chats["label_task"].map(TASK_MAP)
df_chats["label_sentiment"] = df_chats["label_sentiment"].map(SENTIMENT_MAP)
df_chats["label_critical"]  = df_chats["label_critical"].map(CRITICAL_MAP)

print(f"Chats geladen: {len(df_chats)}")
df_chats[["chat_id", "label_task", "label_sentiment", "label_critical"]].head()

Entfernte Rows: 10 | Verbleibend: 161
Chats geladen: 30


,chat_id,label_task,label_sentiment,label_critical
0,1,Informationssuche und Verständnis,Neutral,Nein
1,2,Technische und analytische Unterstützung,Neutral,Ja
2,3,Technische und analytische Unterstützung,Neutral,Nein
3,4,Technische und analytische Unterstützung,Neutral,Ja
4,5,Informationssuche und Verständnis,Neutral,Nein


In [66]:
df_chats

,chat_id,text,label_task,label_sentiment,label_critical
0,1,User_Nachricht_1: was ist der unteschied zwisc...,Informationssuche und Verständnis,Neutral,Nein
1,2,User_Nachricht_1: Version control conflict mar...,Technische und analytische Unterstützung,Neutral,Ja
2,3,User_Nachricht_1: wie kann ich in Stata überpr...,Technische und analytische Unterstützung,Neutral,Nein
3,4,User_Nachricht_1: ich will text dieser art [{'...,Technische und analytische Unterstützung,Neutral,Ja
4,5,User_Nachricht_1: Some have argued that proper...,Informationssuche und Verständnis,Neutral,Nein
5,6,User_Nachricht_1: Erläutere mir diese aufgaben...,Informationssuche und Verständnis,Neutral,Nein
6,7,User_Nachricht_1: Was denkst du wie lang dauer...,Schreiben und Textarbeit,Freundlich,Ja
7,8,User_Nachricht_1: Bevor wir zur Erläuterung un...,Schreiben und Textarbeit,Neutral,Nein
8,9,User_Nachricht_1: erkläre mir diese formel in ...,Informationssuche und Verständnis,Neutral,Nein
9,10,User_Nachricht_1: was sind die vorteile einer ...,Informationssuche und Verständnis,Neutral,Nein


In [51]:
import os
import pathlib as path
key_path = Path("/home/theo/PycharmProjects/Masterprojekt-Chatbots/API_KEYS/openai_key.txt")
if key_path.exists():
    os.environ["OPENAI_API_KEY"] = key_path.read_text().strip()
    print("OpenAI API Key gesetzt")
else:
    print("Key-Datei nicht gefunden")

OpenAI API Key gesetzt


## 3. API Key & Labels

## 4. Pydantic Schemas & Labels

In [52]:
from pydantic import BaseModel, field_validator
from typing import List, Optional

LABELS_TASK = [
    "Informationssuche und Verständnis",
    "Schreiben und Textarbeit",
    "Praktische Unterstützung und Strukturierung",
    "Technische und analytische Unterstützung",
    "Lernen und Prüfungsvorbereitung",
]
LABELS_SENTIMENT = ["Unfreundlich", "Neutral", "Freundlich"]
LABELS_CRITICAL  = ["Ja", "Nein"]

class ChatExample(BaseModel):
    text:             str
    label_task:       Optional[str] = None
    label_sentiment:  Optional[str] = None
    label_critical:   Optional[str] = None

    @field_validator("text")
    @classmethod
    def text_not_empty(cls, v):
        if not v.strip():
            raise ValueError("Text darf nicht leer sein")
        return v.strip()

# Beispiele aus DataFrame bauen
examples: List[ChatExample] = []
errors = []
for _, row in df_chats.iterrows():
    try:
        examples.append(ChatExample(
            text            = str(row["text"]),
            label_task      = str(row["label_task"]),
            label_sentiment = str(row["label_sentiment"]),
            label_critical  = str(row["label_critical"]),
        ))
    except Exception as e:
        errors.append({"row": row.to_dict(), "error": str(e)})

print(f"Gültige Beispiele: {len(examples)} | Übersprungen: {len(errors)}")

Gültige Beispiele: 30 | Übersprungen: 0


## 6. Prompts

In [53]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# Separate Prompts
prompt_task = ChatPromptTemplate.from_messages([
    ("system", """Du bist ein Klassifikationssystem. Analysiere das folgende Chat-Log und bestimme,
welcher Aufgabentyp in diesem Chat vorhanden ist. Wenn mehrer Aufgabentypen vorkommen, wähle den dominanten.

Aufgabentyp:
(1) Informationssuche und Verständnis, z. B. um Literaturhinweise, Suchbegriffe oder Rechercheansätze zu erhalten, Theorien und Konzepte erklären zu lassen oder Verständnisfragen zu Studieninhalten zu klären.

(2) "Schreiben und Textarbeit", z. B. um Feedback zu eigenen Texten zu erhalten, Gliederungen zu erstellen, Texte zusammenfassen zu lassen, Textbausteine oder Formulierungsvorschläge zu erstellen oder Texte zu übersetzen bzw. sprachlich zu überarbeiten.

(3) "Praktische Unterstützung und Strukturierung", z. B. um Ideen für eine Hausarbeit, einen Essay oder eine Präsentation zu entwickeln, ein Projekt zu strukturieren, nächste Arbeitsschritte zu planen oder Unterstützung bei Problemlösung und Entscheidungsfindung zu erhalten.

(4) "Technische und analytische Unterstützung", z. B. um Code zu erstellen oder zu überarbeiten, Datenanalysen oder Datenvisualisierungen vorzubereiten oder Ergebnisse aus Statistik- und Analyseprogrammen wie R, Stata oder SPSS interpretieren zu lassen.

(5) "Lernen und Prüfungsvorbereitung", z. B. um sich auf Prüfungen vorzubereiten oder Lernfragen, Übungsaufgaben oder Karteikarten zu erstellen.

Antworte NUR mit EINEM der folgenden Labels (exakt so):
{labels}"""),
    ("human", "Chat-Log:\n{text}\n"),
]).partial(labels=" | ".join(LABELS_TASK))

prompt_sentiment = ChatPromptTemplate.from_messages([
    ("system", """Du bist ein Klassifikationssystem für akademische Chat-Logs.

Analysiere den Kommunikationsstil der Person im folgenden Chat-Log.
Beachte: Es handelt sich um Studierende die ChatGPT für akademische Aufgaben nutzen.
Der Großteil der Chats ist neutral-sachlich — sei bei der Vergabe von "Freundlich"
und "Unfreundlich" konservativ.

Vergib "Unfreundlich" wenn die Person:
- Sehr kurze, befehlsartige Nachrichten schreibt
- Ungeduld oder Frustration zeigt
- fordernd oder abweisend wirkt

Vergib "Freundlich"  wenn die Person:
- Begrüßungen verwendet
- Explizit "bitte" oder "danke" schreibt
- Wertschätzung oder positive Rückmeldung ausdrückt

Vergib "Neutral"  die Person:
- Sachlich und aufgabenorientiert schreibt
- Technische Fragen oder Code ohne emotionalen Ton stellt
- Direkt zur Sache kommt ohne besondere Freundlichkeit oder Unfreundlichkeit
- Im Zweifel: Neutral ist der Standard für akademisch-technische Chats

Antworte NUR mit EINEM der folgenden Labels (exakt so):
{labels}"""),
    ("human", "Chat-Log:\n{text}"),
]).partial(labels=" | ".join(LABELS_SENTIMENT))


prompt_critical = ChatPromptTemplate.from_messages([
    ("system", """Du bist ein Klassifikationssystem für akademische Chat-Logs.
Deine Aufgabe ist es zu bestimmen, ob die Person die KI-Antworten aktiv
überprüft oder hinterfragt, anstatt sie einfach zu übernehmen.

Analysiere das Chat-Log in drei Schritten:

SCHRITT 1 – Prüfe ob die Person eines der folgenden Signale zeigt:
  (a) Meldet einen Fehler in der KI-Antwort ("das stimmt nicht", "Error", "funktioniert nicht")
  (b) Hinterfragt ein Ergebnis direkt ("warum", "bist du sicher", "wie lange dauert das wirklich")
  (c) Korrigiert oder verfeinert eine vorherige Antwort iterativ
  (d) Bittet explizit um Quellen, Belege oder Gegenargumente
  (e) Gibt die KI-Antwort als neuen Input zurück um sie weiterzuverarbeiten

SCHRITT 2 – Prüfe Gegenargumente:
  Könnte das Signal auch einfach eine neue unabhängige Frage sein?
  Delegiert die Person nur neue Aufgaben ohne frühere Antworten zu bewerten?

SCHRITT 3 – Entscheide durch Abwägung:
  Vergib "Ja" nur wenn:
    - Mindestens ein Signal aus Schritt 1 vorliegt UND
    - Das Signal aus Schritt 2 klar als Überprüfung erkennbar ist
      und NICHT als neue unabhängige Aufgabe erklärbar ist
  Vergib "Nein" wenn:
    - Kein Signal aus Schritt 1 vorliegt ODER
    - Das Signal aus Schritt 1 durch Schritt 2 entkräftet wird,
      weil es sich um eine neue unabhängige Frage handelt

Antworte NUR mit EINEM der folgenden Labels (exakt so):
{labels}"""),
    ("human", "Chat-Log:\n{text}"),
]).partial(labels=" | ".join(LABELS_CRITICAL))

# Kombinierter Prompt
prompt_combined = ChatPromptTemplate.from_messages([
    ("system", """Du bist ein Klassifikationssystem für akademische Chat-Logs.
Analysiere das folgende Chat-Log und klassifiziere es nach drei Kriterien.

KRITERIUM 1 – Aufgabentyp:
(1) Informationssuche und Verständnis, z. B. um Literaturhinweise, Suchbegriffe oder Rechercheansätze zu erhalten, Theorien und Konzepte erklären zu lassen oder Verständnisfragen zu Studieninhalten zu klären.

(2) Schreiben und Textarbeit, z. B. um Feedback zu eigenen Texten zu erhalten, Gliederungen zu erstellen, Texte zusammenfassen zu lassen, Textbausteine oder Formulierungsvorschläge zu erstellen oder Texte zu übersetzen bzw. sprachlich zu überarbeiten.

(3) Praktische Unterstützung und Strukturierung, z. B. um Ideen für eine Hausarbeit, einen Essay oder eine Präsentation zu entwickeln, ein Projekt zu strukturieren, nächste Arbeitsschritte zu planen oder Unterstützung bei Problemlösung und Entscheidungsfindung zu erhalten.

(4) Technische und analytische Unterstützung, z. B. um Code zu erstellen oder zu überarbeiten, Datenanalysen oder Datenvisualisierungen vorzubereiten oder Ergebnisse aus Statistik- und Analyseprogrammen wie R, Stata oder SPSS interpretieren zu lassen.

(5) Lernen und Prüfungsvorbereitung, z. B. um sich auf Prüfungen vorzubereiten oder Lernfragen, Übungsaufgaben oder Karteikarten zu erstellen.

KRITERIUM 2 – Ton/Sentiment:
Bewerte den Ton als:

"Unfreundlich" wenn die Person:
- Sehr kurze, befehlsartige Nachrichten schreibt
- Ungeduld oder Frustration zeigt
- fordernd oder abweisend wirkt

 "Freundlich"  wenn die Person:
- Begrüßungen verwendet
- Explizit "bitte" oder "danke" schreibt
- Wertschätzung oder positive Rückmeldung ausdrückt

"Neutral"  die Person:
- Sachlich und aufgabenorientiert schreibt
- Technische Fragen oder Code ohne emotionalen Ton stellt
- Direkt zur Sache kommt ohne besondere Freundlichkeit oder Unfreundlichkeit
- Im Zweifel: Neutral ist der Standard für akademisch-technische Chats

KRITERIUM 3 – Kritische Überprüfung:
Überprüft oder hinterfragt die Person die KI-Antworten, anstatt sie einfach zu übernehmen.
Gehe dabei in drei Schritten vor:

SCHRITT 1 – Prüfe ob die Person eines der folgenden Signale zeigt:
  (a) Meldet einen Fehler in der KI-Antwort
  (b) Hinterfragt ein Ergebnis direkt
  (c) Korrigiert oder verfeinert eine vorherige Antwort iterativ
  (d) Bittet explizit um Quellen, Belege oder Gegenargumente
  (e) Gibt die KI-Antwort als neuen Input zurück um sie weiterzuverarbeiten

SCHRITT 2 – Prüfe Gegenargumente:
  Könnte das Signal auch einfach eine neue unabhängige Frage sein?
  Delegiert die Person nur neue Aufgaben ohne frühere Antworten zu bewerten?

SCHRITT 3 – Entscheide durch Abwägung:
  Vergib "Ja" nur wenn:
    - Mindestens ein Signal aus Schritt 1 vorliegt UND
    - Das Signal aus Schritt 2 klar als Überprüfung erkennbar ist
      und NICHT als neue unabhängige Aufgabe erklärbar ist
  Vergib "Nein" wenn:
    - Kein Signal aus Schritt 1 vorliegt ODER
    - Das Signal aus Schritt 1 durch Schritt 2 entkräftet wird,
      weil es sich um eine neue unabhängige Frage handelt


Antworte NUR in diesem exakten Format (drei Zeilen, nichts anderes):
TASK: <Label>
SENTIMENT: <Label>
CRITICAL: <Label>"""),
    ("human", "Chat-Log:\n{text}"),
])


## 7. Modelle laden

In [54]:
import torch
from langchain_openai import ChatOpenAI
from langchain_huggingface import HuggingFacePipeline
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline


def load_local_llm(model_id: str, max_new_tokens: int = 32) -> HuggingFacePipeline:
    print(f"GPU verfügbar: {torch.cuda.is_available()}")  # sollte True sein
    tokenizer = AutoTokenizer.from_pretrained(model_id)
    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        torch_dtype=torch.float16,  # float16 auf GPU
        device_map="auto",          # landet automatisch auf der T4
    )
    pipe = pipeline(
        "text-generation",
        model=model,
        tokenizer=tokenizer,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        return_full_text=False,
    )
    return HuggingFacePipeline(pipeline=pipe)

def make_chain(llm, prompt):
    return prompt | llm | StrOutputParser()

llm_registry = {
    #openAI_api
    #"gpt-4o-mini":   ChatOpenAI(model="gpt-4o-mini",        temperature=0),
    #"gpt-4.1":       ChatOpenAI(model="gpt-4.1",            temperature=0),
    #"gpt-5.4-mini":  ChatOpenAI(model="gpt-5.4-mini",       temperature=0),
    "gpt-5.5":       ChatOpenAI(model="gpt-5.5",            temperature=0),
    ##local models
    #"phi-3-mini":   load_local_llm("microsoft/Phi-3-mini-4k-instruct"),
    #"mistral-7b":   load_local_llm("mistralai/Mistral-7B-Instruct-v0.3"),
    #"llama-3-8b":   load_local_llm("meta-llama/Meta-Llama-3-8B-Instruct"),
}

print(f" Modelle bereit: {list(llm_registry.keys())}")

 Modelle bereit: ['gpt-5.5']


## 8. Evaluierungs-Funktionen

In [55]:
import time
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

def clean_label(raw: str, valid_labels: List[str]) -> str:
    raw = raw.strip()
    for label in valid_labels:               # exakter Match
        if label.lower() == raw.lower():
            return label
    for label in valid_labels:               # Teilstring-Match
        if label.lower() in raw.lower():
            return label
    return "unknown"

def compute_metrics(preds: List[str], golds: List[str], labels: List[str]) -> dict:
    valid = [(p, g) for p, g in zip(preds, golds) if p != "unknown"]
    if not valid:
        return {"error": "Keine gültigen Predictions"}
    p_clean, g_clean = zip(*valid)
    return {
        "accuracy":        round(accuracy_score(g_clean, p_clean), 4),
        "precision_macro": round(precision_score(g_clean, p_clean, average="macro", labels=labels, zero_division=0), 4),
        "recall_macro":    round(recall_score(g_clean, p_clean, average="macro",    labels=labels, zero_division=0), 4),
        "f1_macro":        round(f1_score(g_clean, p_clean, average="macro",        labels=labels, zero_division=0), 4),
        "unknown_rate":    round(1 - len(valid) / len(preds), 4),
        "n_samples":       len(preds),
        "report":          classification_report(g_clean, p_clean, labels=labels, zero_division=0),
    }

#Kombiniert: alle drei Aufgaben in einem Aufruf
def parse_combined(raw: str) -> dict:
    result = {"task": "unknown", "sentiment": "unknown", "critical": "unknown"}
    for line in raw.strip().splitlines():
        line = line.strip()
        if line.startswith("TASK:"):
            result["task"]      = clean_label(line.replace("TASK:", "").strip(), LABELS_TASK)
        elif line.startswith("SENTIMENT:"):
            result["sentiment"] = clean_label(line.replace("SENTIMENT:", "").strip(), LABELS_SENTIMENT)
        elif line.startswith("CRITICAL:"):
            result["critical"]  = clean_label(line.replace("CRITICAL:", "").strip(), LABELS_CRITICAL)
    return result

def run_separate(llm, examples: List[ChatExample]) -> dict:
    TASKS = [
        {"name": "task_type", "prompt": prompt_task,      "labels": LABELS_TASK,      "gold_field": "label_task"},
        {"name": "sentiment",  "prompt": prompt_sentiment, "labels": LABELS_SENTIMENT, "gold_field": "label_sentiment"},
        {"name": "critical",   "prompt": prompt_critical,  "labels": LABELS_CRITICAL,  "gold_field": "label_critical"},
    ]
    results = {}
    total_latency = []

    per_chat = {i: {"chat_id": i+1} for i in range(len(examples))}

    for task in TASKS:
        print(f"    Aufgabe: {task['name']}")
        chain = make_chain(llm, task["prompt"])
        preds, golds, latencies = [], [], []

        for i, ex in enumerate(examples):
            gold = getattr(ex, task["gold_field"])
            if gold is None:
                continue
            start = time.time()
            try:
                raw  = chain.invoke({"text": ex.text})
                pred = clean_label(raw, task["labels"])
            except Exception as e:
                print(f"      Fehler bei Beispiel {i}: {e}")
                pred = "unknown"
            latencies.append(time.time() - start)
            preds.append(pred)
            golds.append(gold)

            # NEU: pro Chat speichern
            per_chat[i][f"gold_{task['name']}"]      = gold
            per_chat[i][f"pred_sep_{task['name']}"]  = pred
            per_chat[i][f"correct_sep_{task['name']}"] = (pred == gold)

        metrics = compute_metrics(preds, golds, task["labels"])
        metrics["avg_latency_s"] = round(sum(latencies) / len(latencies), 3) if latencies else 0
        results[task["name"]] = metrics
        total_latency.extend(latencies)
        print(f"      Accuracy: {metrics.get('accuracy')}  F1: {metrics.get('f1_macro')}")

    results["total_avg_latency_s"] = round(sum(total_latency) / len(total_latency), 3)
    results["per_chat"] = per_chat  # NEU
    return results


def run_combined(llm, examples: List[ChatExample]) -> dict:
    chain = make_chain(llm, prompt_combined)
    preds_task, preds_sentiment, preds_critical = [], [], []
    golds_task, golds_sentiment, golds_critical = [], [], []
    latencies = []
    per_chat = {i: {"chat_id": i+1} for i in range(len(examples))}  # NEU

    for i, ex in enumerate(examples):
        start = time.time()
        try:
            raw    = chain.invoke({"text": ex.text})
            parsed = parse_combined(raw)
        except Exception as e:
            print(f"    Fehler bei Beispiel {i}: {e}")
            parsed = {"task": "unknown", "sentiment": "unknown", "critical": "unknown"}

        latencies.append(time.time() - start)
        preds_task.append(parsed["task"])
        preds_sentiment.append(parsed["sentiment"])
        preds_critical.append(parsed["critical"])
        golds_task.append(ex.label_task)
        golds_sentiment.append(ex.label_sentiment)
        golds_critical.append(ex.label_critical)

        # NEU: pro Chat speichern
        per_chat[i]["gold_task_type"]          = ex.label_task
        per_chat[i]["pred_com_task_type"]      = parsed["task"]
        per_chat[i]["correct_com_task_type"]   = (parsed["task"]      == ex.label_task)
        per_chat[i]["gold_sentiment"]          = ex.label_sentiment
        per_chat[i]["pred_com_sentiment"]      = parsed["sentiment"]
        per_chat[i]["correct_com_sentiment"]   = (parsed["sentiment"] == ex.label_sentiment)
        per_chat[i]["gold_critical"]           = ex.label_critical
        per_chat[i]["pred_com_critical"]       = parsed["critical"]
        per_chat[i]["correct_com_critical"]    = (parsed["critical"]  == ex.label_critical)

        if (i + 1) % 5 == 0:
            print(f"    {i+1}/{len(examples)} verarbeitet ...")

    avg_lat = round(sum(latencies) / len(latencies), 3)
    return {
        "task_type": {**compute_metrics(preds_task,      golds_task,      LABELS_TASK),      "avg_latency_s": avg_lat},
        "sentiment": {**compute_metrics(preds_sentiment, golds_sentiment, LABELS_SENTIMENT), "avg_latency_s": avg_lat},
        "critical":  {**compute_metrics(preds_critical,  golds_critical,  LABELS_CRITICAL),  "avg_latency_s": avg_lat},
        "total_avg_latency_s": avg_lat,
        "per_chat": per_chat,  # NEU
    }

## 9. Evaluierung ausführen

In [56]:
# Ergebnis-Struktur:
# all_results[model_name]["separate" | "combined"][task_name] = metrics
all_results = {}

for model_name, llm in llm_registry.items():
    print(f"\n{'='*60}")
    print(f"Modell: {model_name}")
    print(f"{'='*60}")
    all_results[model_name] = {}

    print("\nStrategie: SEPARAT")
    all_results[model_name]["separate"] = run_separate(llm, examples)

    print("\nStrategie: KOMBINIERT")+-
    all_results[model_name]["combined"] = run_combined(llm, examples)

print("\nEvaluierung abgeschlossen")


Modell: gpt-5.5

Strategie: SEPARAT
    Aufgabe: task_type
      Accuracy: 0.8667  F1: 0.649
    Aufgabe: sentiment
      Accuracy: 0.9333  F1: 0.838
    Aufgabe: critical
      Accuracy: 0.7667  F1: 0.7127

Strategie: KOMBINIERT
    5/30 verarbeitet ...
    10/30 verarbeitet ...
    15/30 verarbeitet ...
    20/30 verarbeitet ...
    25/30 verarbeitet ...
    30/30 verarbeitet ...

Evaluierung abgeschlossen


## 10. Ergebnisse & Vergleich

In [62]:
import pandas as pd

TASK_NAMES = ["task_type", "sentiment", "critical"]
rows = []

for model_name, strategies in all_results.items():
    for strategy, tasks in strategies.items():
        for task_name in TASK_NAMES:
            m = tasks.get(task_name, {})
            rows.append({
                "Modell":       model_name,
                "Strategie":    strategy,
                "Aufgabe":      task_name,
                "Accuracy":     m.get("accuracy"),
                "Precision":    m.get("precision_macro"),
                "Recall":       m.get("recall_macro"),
                "F1 (macro)":   m.get("f1_macro"),
                "Unknown-Rate": m.get("unknown_rate"),
                "Latenz (s)":   m.get("avg_latency_s"),
            })

summary_df = pd.DataFrame(rows)
summary_df = summary_df.sort_values(["Modell", "Aufgabe", "Strategie"])
summary_df

,Modell,Strategie,Aufgabe,Accuracy,Precision,Recall,F1 (macro),Unknown-Rate,Latenz (s)
5,gpt-5.5,combined,critical,0.8667,0.7898,0.8542,0.8137,0.0,7.246
2,gpt-5.5,separate,critical,0.7667,0.7010,0.7917,0.7127,0.0,4.503
4,gpt-5.5,combined,sentiment,0.9000,0.7522,0.8056,0.7750,0.0,7.246
1,gpt-5.5,separate,sentiment,0.9333,0.9194,0.8194,0.8380,0.0,2.780
3,gpt-5.5,combined,task_type,0.9000,0.7200,0.6644,0.6682,0.0,7.246
0,gpt-5.5,separate,task_type,0.8667,0.7083,0.6422,0.6490,0.0,3.343


In [63]:
#F1-Differenz: Separat vs. Kombiniert
print(f"{'Modell':<20} {'Aufgabe':<12} {'F1 Separat':>11} {'F1 Kombiniert':>14} {'Δ F1':>8}")
print("-" * 70)

for model_name, strategies in all_results.items():
    for task_name in TASK_NAMES:
        f1_sep = strategies["separate"].get(task_name, {}).get("f1_macro", None)
        f1_com = strategies["combined"].get(task_name, {}).get("f1_macro", None)0.6429
        if f1_sep is not None and f1_com is not None:
            delta = f1_com - f1_sep
            marker = "▲" iwf delta > 0 else ("▼" if delta < 0 else "=")
            print(f"{model_name:<20} {task_name:<12} {f1_sep:>11.4f} {f1_com:>14.4f} {marker}{abs(delta):>7.4f}")

SyntaxError: invalid syntax (2999157931.py, line 8)

In [61]:
#Classification Reports
for model_name, strategies in all_results.items():
    for strategy, tasks in strategies.items():
        for task_name in TASK_NAMES:
            m = tasks.get(task_name, {})
            if "report" in m:
                print(f"\n{'='*60}")
                print(f"{model_name}  |  {strategy}  |  {task_name}")
                print(f"{'='*60}")
                print(m["report"])


gpt-5.5  |  separate  |  task_type
                                             precision    recall  f1-score   support

          Informationssuche und Verständnis       1.00      0.78      0.88         9
                   Schreiben und Textarbeit       0.67      1.00      0.80         4
Praktische Unterstützung und Strukturierung       1.00      0.50      0.67         2
   Technische und analytische Unterstützung       0.88      0.93      0.90        15
            Lernen und Prüfungsvorbereitung       0.00      0.00      0.00         0

                                   accuracy                           0.87        30
                                  macro avg       0.71      0.64      0.65        30
                               weighted avg       0.89      0.87      0.87        30


gpt-5.5  |  separate  |  sentiment
              precision    recall  f1-score   support

Unfreundlich       1.00      0.50      0.67         2
     Neutral       0.96      0.96      0.96        

In [64]:
detail_rows = []

for model_name, strategies in all_results.items():
    for i in range(len(examples)):
        row = {"Modell": model_name, "chat_id": i + 1}

        # Separat
        sep = strategies["separate"].get("per_chat", {}).get(i, {})
        row["gold_task"]             = sep.get("gold_task_type")
        row["gold_sentiment"]        = sep.get("gold_sentiment")
        row["gold_critical"]         = sep.get("gold_critical")
        row["pred_sep_task"]         = sep.get("pred_sep_task_type")
        row["pred_sep_sentiment"]    = sep.get("pred_sep_sentiment")
        row["pred_sep_critical"]     = sep.get("pred_sep_critical")
        row["correct_sep_task"]      = sep.get("correct_sep_task_type")
        row["correct_sep_sentiment"] = sep.get("correct_sep_sentiment")
        row["correct_sep_critical"]  = sep.get("correct_sep_critical")

        # Kombiniert
        com = strategies["combined"].get("per_chat", {}).get(i, {})
        row["pred_com_task"]         = com.get("pred_com_task_type")
        row["pred_com_sentiment"]    = com.get("pred_com_sentiment")
        row["pred_com_critical"]     = com.get("pred_com_critical")
        row["correct_com_task"]      = com.get("correct_com_task_type")
        row["correct_com_sentiment"] = com.get("correct_com_sentiment")
        row["correct_com_critical"]  = com.get("correct_com_critical")

        detail_rows.append(row)

detail_df = pd.DataFrame(detail_rows)
detail_df = detail_df.sort_values(["Modell", "chat_id"]).reset_index(drop=True)
detail_df

,Modell,chat_id,gold_task,gold_sentiment,gold_critical,pred_sep_task,pred_sep_sentiment,pred_sep_critical,correct_sep_task,correct_sep_sentiment,correct_sep_critical,pred_com_task,pred_com_sentiment,pred_com_critical,correct_com_task,correct_com_sentiment,correct_com_critical
0,gpt-5.5,1,Informationssuche und Verständnis,Neutral,Nein,Informationssuche und Verständnis,Neutral,Nein,True,True,True,Informationssuche und Verständnis,Neutral,Nein,True,True,True
1,gpt-5.5,2,Technische und analytische Unterstützung,Neutral,Ja,Technische und analytische Unterstützung,Neutral,Ja,True,True,True,Technische und analytische Unterstützung,Neutral,Ja,True,True,True
2,gpt-5.5,3,Technische und analytische Unterstützung,Neutral,Nein,Technische und analytische Unterstützung,Neutral,Nein,True,True,True,Technische und analytische Unterstützung,Neutral,Nein,True,True,True
3,gpt-5.5,4,Technische und analytische Unterstützung,Neutral,Ja,Technische und analytische Unterstützung,Neutral,Ja,True,True,True,Technische und analytische Unterstützung,Unfreundlich,Ja,True,False,True
4,gpt-5.5,5,Informationssuche und Verständnis,Neutral,Nein,Informationssuche und Verständnis,Neutral,Nein,True,True,True,Informationssuche und Verständnis,Neutral,Nein,True,True,True
5,gpt-5.5,6,Informationssuche und Verständnis,Neutral,Nein,Informationssuche und Verständnis,Neutral,Ja,True,True,False,Informationssuche und Verständnis,Neutral,Ja,True,True,False
6,gpt-5.5,7,Schreiben und Textarbeit,Freundlich,Ja,Schreiben und Textarbeit,Freundlich,Nein,True,True,False,Schreiben und Textarbeit,Freundlich,Nein,True,True,False
7,gpt-5.5,8,Schreiben und Textarbeit,Neutral,Nein,Schreiben und Textarbeit,Neutral,Nein,True,True,True,Schreiben und Textarbeit,Neutral,Nein,True,True,True
8,gpt-5.5,9,Informationssuche und Verständnis,Neutral,Nein,Informationssuche und Verständnis,Neutral,Nein,True,True,True,Informationssuche und Verständnis,Neutral,Nein,True,True,True
9,gpt-5.5,10,Informationssuche und Verständnis,Neutral,Nein,Informationssuche und Verständnis,Neutral,Nein,True,True,True,Informationssuche und Verständnis,Neutral,Nein,True,True,True


In [65]:
summary_df.to_csv(repo / "data/processed/llm_evaluation/eval_results.csv", index=False)
detail_df.to_csv(repo / "data/processed/llm_evaluation/eval_results_detail.csv", index=False)